# LeetCode 1191 — K-Concatenation Maximum Sum bằng Python

Link đề: https://leetcode.com/problems/k-concatenation-maximum-sum/ · Độ khó: **Medium** · Chủ đề: Array, Dynamic Programming

Skeleton LeetCode:

```python
class Solution:
    def kConcatenationMaxSum(self, arr: list[int], k: int) -> int:
        ...
```

Ghi chú: mảng được **lặp lại `k` lần** (có thể dài tới `10^10` phần tử) nên không thể xây thật. Ý chính: đoạn con tối ưu hoặc nằm gọn trong **≤ 2 bản sao**, hoặc là **hậu tố của bản đầu + (k−2) bản đầy đủ + tiền tố của bản cuối** — chỉ có lợi khi tổng một bản `> 0`. Đoạn con **được phép rỗng** (tổng `0`), và đáp án lấy **modulo `10^9 + 7`**.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: tìm tổng subarray lớn nhất (cho phép rỗng) trên mảng `arr` nối `k` lần, trả về modulo `10^9+7`.
2. Ôn **Kadane** phiên bản cho phép rỗng: `cur = max(0, cur + x)`.
3. Lập luận cấu trúc: đoạn tối ưu "trải" qua các bản sao như thế nào; vì sao **2 bản sao là đủ** khi tổng âm/bằng 0.
4. Kết hợp **tổng tiền tố lớn nhất / hậu tố lớn nhất** với số bản sao ở giữa.
5. Tránh bẫy: lấy modulo **giữa chừng** rồi so sánh `max` (sai), quên `k = 1`, xây mảng `k·n` phần tử (tràn bộ nhớ/TLE).

## 2. Đề bài & ví dụ

Cho mảng số nguyên `arr` và số nguyên `k`. Tạo mảng mới bằng cách **lặp lại** `arr` đúng `k` lần (ví dụ `arr = [1, 2]`, `k = 3` → `[1, 2, 1, 2, 1, 2]`).

Trả về **tổng lớn nhất** của một subarray trong mảng mới. Subarray **có thể có độ dài 0**, khi đó tổng là `0`. Vì đáp án có thể rất lớn, trả về modulo `10^9 + 7`.

**Ví dụ 1:** `arr = [1, 2]`, `k = 3` → `9` (lấy cả mảng).

**Ví dụ 2:** `arr = [1, -2, 1]`, `k = 5` → `2` (đoạn `[1, 1]` nối giữa hai bản sao).

**Ví dụ 3:** `arr = [-1, -2]`, `k = 7` → `0` (chọn đoạn rỗng).

**Ràng buộc:** `1 <= arr.length <= 10^5`, `1 <= k <= 10^5`, `-10^4 <= arr[i] <= 10^4`.

In [1]:
import random
import time
from itertools import accumulate
from typing import List

MOD = 10**9 + 7
examples = [
    ([1, 2], 3, 9),
    ([1, -2, 1], 5, 2),
    ([-1, -2], 7, 0),
]
for arr, k, want in examples:
    print(f"arr={arr}, k={k} -> mảng nối: {arr * k} -> {want}")

arr=[1, 2], k=3 -> mảng nối: [1, 2, 1, 2, 1, 2] -> 9
arr=[1, -2, 1], k=5 -> mảng nối: [1, -2, 1, 1, -2, 1, 1, -2, 1, 1, -2, 1, 1, -2, 1] -> 2
arr=[-1, -2], k=7 -> mảng nối: [-1, -2, -1, -2, -1, -2, -1, -2, -1, -2, -1, -2, -1, -2] -> 0


## 3. Kiến thức nền

### 3.1 Nhân list `arr * k`

`[1, 2] * 3 == [1, 2, 1, 2, 1, 2]`. Tiện cho brute force với input nhỏ, nhưng với `n = k = 10^5` sẽ tạo `10^10` phần tử — **không bao giờ** làm vậy trong lời giải thật.

In [2]:
print([1, -2, 1] * 2)
print(len([0] * 10**3 * 10**2))   # 10^5 phần tử thì ổn; 10^10 thì không

[1, -2, 1, 1, -2, 1]
100000


### 3.2 Kadane cho phép đoạn rỗng

Khi đoạn rỗng hợp lệ (tổng `0`), Kadane trở thành: `cur = max(0, cur + x)`, `best = max(best, cur)` với `best` khởi tạo `0`. Nếu `cur` âm thì "vứt bỏ" và bắt đầu lại từ rỗng.

In [3]:
def kadane0(a):
    cur = best = 0
    for x in a:
        cur = max(0, cur + x)
        best = max(best, cur)
    return best


print(kadane0([1, -2, 1]), kadane0([-1, -2]), kadane0([1, -2, 1, 1, -2, 1]))

1 0 2


### 3.3 Tiền tố lớn nhất / hậu tố lớn nhất

- **Tiền tố (prefix)** `arr[0..i]`; tổng tiền tố lớn nhất (cho phép rỗng) = `max(0, max(accumulate(arr)))`.
- **Hậu tố (suffix)** `arr[i..n-1]`; tính bằng accumulate trên mảng đảo ngược.

In [4]:
arr = [1, -2, 1]
best_prefix = max(0, max(accumulate(arr)))
best_suffix = max(0, max(accumulate(reversed(arr))))
print(list(accumulate(arr)), best_prefix, best_suffix)

[1, -1, 0] 1 1


### 3.4 Modulo và số nguyên lớn trong Python

Python có số nguyên **không giới hạn**, nên `(k - 2) * sum(arr)` (tới `10^5 · 10^9 = 10^14`) không tràn. Ta chỉ lấy `% MOD` **ở bước cuối cùng**. (Trong C++/Java phải dùng `long long`/`long`.) Quan trọng: **không được** lấy modulo trước khi so sánh `max`, vì modulo phá vỡ thứ tự lớn/nhỏ.

In [5]:
a, b = 10**9 + 8, 5
print(max(a, b) % MOD, max(a % MOD, b % MOD))   # khác nhau!

1 5


## 4. Ý tưởng nền tảng / trực giác

Xét `arr = [1, -2, 1]` (tổng `S = 0`), `k = 5`. Nối 2 bản: `[1, -2, 1, 1, -2, 1]` → đoạn `[1, 1]` (vắt qua chỗ nối) tổng `2`. Thêm bản thứ 3, 4, 5 không giúp được gì vì mỗi bản đầy đủ đóng góp `S = 0`.

Xét `arr = [1, 2]` (tổng `S = 3 > 0`), `k = 3`: nên lấy **mọi** bản sao — mỗi bản thêm `+3`.

Tổng quát, một đoạn trong mảng nối có dạng:
1. Nằm gọn trong **1 bản**, hoặc
2. **Hậu tố** của một bản + `m` bản đầy đủ (`m ≥ 0`) + **tiền tố** của bản tiếp theo.

Với dạng 2: nếu `S > 0` thì `m` càng lớn càng tốt → `m = k - 2`; nếu `S ≤ 0` thì `m = 0` (chỉ cần 2 bản).

In [6]:
for arr, k in [([1, -2, 1], 5), ([1, 2], 3), ([2, -5, 3], 4)]:
    S = sum(arr)
    print(f"arr={arr} S={S}: kadane0 trên 1, 2, 3 bản =", [kadane0(arr * m) for m in (1, 2, 3)])

arr=[1, -2, 1] S=0: kadane0 trên 1, 2, 3 bản = [1, 2, 2]
arr=[1, 2] S=3: kadane0 trên 1, 2, 3 bản = [3, 6, 9]
arr=[2, -5, 3] S=0: kadane0 trên 1, 2, 3 bản = [3, 5, 5]


## 5. Cách 1 — Brute Force / baseline

**Ý tưởng:** xây thật mảng `arr * k`, chạy Kadane (cho phép rỗng). Đúng tuyệt đối, dùng làm **oracle** khi `n·k` nhỏ.

In [7]:
def brute(arr: List[int], k: int) -> int:
    return kadane0(arr * k) % MOD


def brute_raw(arr, k):
    """Oracle O((nk)^2): thử mọi đoạn bằng tổng tiền tố."""
    b = arr * k
    P = list(accumulate(b, initial=0))
    best = 0
    for l in range(len(b)):
        for r in range(l + 1, len(b) + 1):
            best = max(best, P[r] - P[l])
    return best % MOD


for arr, k, want in examples:
    print(brute(arr, k), brute_raw(arr, k), want)
    assert brute(arr, k) == brute_raw(arr, k) == want

9 9 9
2 2 2
0 0 0


**Độ phức tạp:** `O(n·k)` thời gian và bộ nhớ. Với `n = k = 10^5` → `10^10` phần tử: **không thể** (cả thời gian lẫn RAM ~ 80GB).

**Nút thắt:** các bản sao giống hệt nhau; ta lặp lại công việc `k` lần. Cần khai thác tính tuần hoàn.

In [8]:
random.seed(0)
arr = [random.randint(-10, 10) for _ in range(1000)]
for k in [10, 100, 1000]:
    t = time.perf_counter(); brute(arr, k); print(f"n=1000, k={k}: {time.perf_counter() - t:.3f}s")

n=1000, k=10: 0.001s
n=1000, k=100: 0.006s
n=1000, k=1000: 0.071s


## 6. Bước đột phá

**Insight:** gọi `S = sum(arr)`.

- **Nếu `k == 1`:** đáp án là `kadane0(arr)`.
- **Nếu `k >= 2` và `S <= 0`:** đoạn tối ưu nằm trong **2 bản sao liền nhau** → `kadane0(arr * 2)`. (Bất kỳ đoạn nào chứa trọn một bản sao ở giữa, bỏ bản đó đi tổng không giảm vì `S <= 0`.)
- **Nếu `k >= 2` và `S > 0`:** đoạn tối ưu = hậu tố tốt nhất + `(k − 2)·S` + tiền tố tốt nhất, tức `kadane0(arr * 2) + (k − 2)·S`.

Gộp: `ans = kadane0(arr * min(k, 2)) + (k − 2)·max(S, 0)` (khi `k >= 2`).

**Invariant cần nhớ:** "bản sao đầy đủ ở giữa" đóng góp đúng `S`; thêm nó có lợi **khi và chỉ khi** `S > 0`.

## 7. Các cách tiếp theo

### Cách 2 — Kadane trên `min(k, 2)` bản + bù `(k−2)·max(S,0)`

In [9]:
def solve_kadane2(arr: List[int], k: int) -> int:
    if k == 1:
        return kadane0(arr) % MOD
    S = sum(arr)
    return (kadane0(arr * 2) + (k - 2) * max(S, 0)) % MOD


for arr, k, want in examples:
    assert solve_kadane2(arr, k) == want
print([solve_kadane2(a, k) for a, k, _ in examples])

[9, 2, 0]


Tại sao khi `S > 0` lại là `kadane0(arr*2) + (k-2)S`? Vì khi `S > 0`, đoạn tối ưu trong 2 bản luôn có thể chọn dạng "hậu tố + tiền tố" (vắt qua chỗ nối) — nếu đoạn tốt nhất nằm gọn trong 1 bản thì `hậu tố tốt nhất + tiền tố tốt nhất ≥ ` nó (sẽ chứng minh ở mục 10). Chèn `k − 2` bản ở giữa đoạn vắt đó cộng thêm `(k − 2)·S`.

**Độ phức tạp:** `O(n)` thời gian, `O(n)` bộ nhớ (do `arr * 2`).

### Cách 3 — Tiền tố / hậu tố tường minh, bộ nhớ `O(1)`

- `best_in = kadane0(arr)` (đoạn trong 1 bản).
- `pre = ` tổng tiền tố lớn nhất (≥ 0), `suf = ` tổng hậu tố lớn nhất (≥ 0).
- Nếu `k ≥ 2`: ứng viên vắt qua = `suf + pre + (k − 2)·max(S, 0)`.
- Đáp án `max(best_in, ứng viên)`.

In [10]:
def solve_prefix_suffix(arr: List[int], k: int) -> int:
    best_in = cur = 0
    pre = run = 0
    S = 0
    for x in arr:
        cur = max(0, cur + x); best_in = max(best_in, cur)
        run += x; pre = max(pre, run)
        S += x
    suf = run = 0
    for x in reversed(arr):
        run += x; suf = max(suf, run)
    if k == 1:
        return best_in % MOD
    return max(best_in, suf + pre + (k - 2) * max(S, 0)) % MOD


for arr, k, want in examples:
    assert solve_prefix_suffix(arr, k) == want
print([solve_prefix_suffix(a, k) for a, k, _ in examples])

[9, 2, 0]


**Độ phức tạp:** `O(n)` thời gian, `O(1)` bộ nhớ.

### Cách 4 — Kadane trên chỉ số ảo `i % n` (không tạo mảng mới)

Chạy Kadane trên `min(k, 2) · n` chỉ số, phần tử thứ `i` là `arr[i % n]`. Giống Cách 2 nhưng không cấp phát `arr * 2`. Kỹ thuật "chỉ số modulo" này rất hay gặp với **mảng vòng**.

In [11]:
def solve_virtual(arr: List[int], k: int) -> int:
    n = len(arr)
    cur = best = 0
    for i in range(n * min(k, 2)):
        cur = max(0, cur + arr[i % n])
        best = max(best, cur)
    if k >= 2:
        best += (k - 2) * max(sum(arr), 0)
    return best % MOD


for arr, k, want in examples:
    assert solve_virtual(arr, k) == want
print([solve_virtual(a, k) for a, k, _ in examples])

[9, 2, 0]


### Cách 5 — Quan sát "Kadane ổn định sau 2 bản" bằng thực nghiệm

Một cách tiếp cận khám phá: chạy Kadane trên `1, 2, 3, ...` bản và nhìn hiệu giữa hai lần liên tiếp. Từ bản thứ 2 trở đi, hiệu luôn bằng `max(S, 0)` — khẳng định công thức tuyến tính theo `k`. (Dùng để **phát hiện** công thức, không phải lời giải chính.)

In [12]:
def kadane_series(arr, upto):
    return [kadane0(arr * m) for m in range(1, upto + 1)]


random.seed(11)
for _ in range(4):
    arr = [random.randint(-6, 6) for _ in range(5)]
    ser = kadane_series(arr, 6)
    diffs = [ser[i + 1] - ser[i] for i in range(len(ser) - 1)]
    print(f"arr={arr} S={sum(arr):>3} kadane={ser} hiệu={diffs}")
    assert all(d == max(sum(arr), 0) for d in diffs[1:])

arr=[1, 2, 6, 1, 1] S= 11 kadane=[11, 22, 33, 44, 55, 66] hiệu=[11, 11, 11, 11, 11]
arr=[2, 3, -3, -4, 6] S=  4 kadane=[6, 11, 15, 19, 23, 27] hiệu=[5, 4, 4, 4, 4]
arr=[2, 1, 4, 3, 6] S= 16 kadane=[16, 32, 48, 64, 80, 96] hiệu=[16, 16, 16, 16, 16]
arr=[-4, -5, 1, -2, -4] S=-14 kadane=[1, 1, 1, 1, 1, 1] hiệu=[0, 0, 0, 0, 0]


## 8. TRACE từng bước

Trace Cách 3 trên `arr = [1, -2, 1]`, `k = 5`:

| x | cur (Kadane) | best_in | run (tiền tố) | pre |
|---|---|---|---|---|
| 1 | 1 | 1 | 1 | 1 |
| -2 | 0 | 1 | -1 | 1 |
| 1 | 1 | 1 | 0 | 1 |

Hậu tố (duyệt ngược `1, -2, 1`): `run = 1, -1, 0` → `suf = 1`. `S = 0`.
Ứng viên vắt = `suf + pre + 3·max(0, 0) = 2`. Đáp án `max(1, 2) = 2`.

In [13]:
def trace_prefix_suffix(arr, k):
    best_in = cur = pre = run = S = 0
    print("Lượt xuôi:")
    for x in arr:
        cur = max(0, cur + x); best_in = max(best_in, cur)
        run += x; pre = max(pre, run); S += x
        print(f"  x={x:>3} cur={cur:>3} best_in={best_in:>3} run={run:>3} pre={pre:>3}")
    suf = run = 0
    print("Lượt ngược (hậu tố):")
    for x in reversed(arr):
        run += x; suf = max(suf, run)
        print(f"  x={x:>3} run={run:>3} suf={suf:>3}")
    cross = suf + pre + (k - 2) * max(S, 0) if k >= 2 else None
    print(f"S={S}, ứng viên trong 1 bản={best_in}, ứng viên vắt qua={cross}")
    return max(best_in, cross if cross is not None else 0) % MOD


print("=>", trace_prefix_suffix([1, -2, 1], 5))
print("=>", trace_prefix_suffix([1, 2], 3))

Lượt xuôi:
  x=  1 cur=  1 best_in=  1 run=  1 pre=  1
  x= -2 cur=  0 best_in=  1 run= -1 pre=  1
  x=  1 cur=  1 best_in=  1 run=  0 pre=  1
Lượt ngược (hậu tố):
  x=  1 run=  1 suf=  1
  x= -2 run= -1 suf=  1
  x=  1 run=  0 suf=  1
S=0, ứng viên trong 1 bản=1, ứng viên vắt qua=2
=> 2
Lượt xuôi:
  x=  1 cur=  1 best_in=  1 run=  1 pre=  1
  x=  2 cur=  3 best_in=  3 run=  3 pre=  3
Lượt ngược (hậu tố):
  x=  2 run=  2 suf=  2
  x=  1 run=  3 suf=  3
S=3, ứng viên trong 1 bản=3, ứng viên vắt qua=9
=> 9


## 9. Code nộp LeetCode

In [14]:
class Solution:
    def kConcatenationMaxSum(self, arr: List[int], k: int) -> int:
        MOD = 10**9 + 7
        best_in = cur = 0          # Kadane (cho phép rỗng) trong 1 bản
        pre = run = 0              # tổng tiền tố lớn nhất
        for x in arr:
            cur = max(0, cur + x)
            best_in = max(best_in, cur)
            run += x
            pre = max(pre, run)
        total = run
        if k == 1:
            return best_in % MOD
        suf = run = 0              # tổng hậu tố lớn nhất
        for x in reversed(arr):
            run += x
            suf = max(suf, run)
        cross = suf + pre + (k - 2) * max(total, 0)
        return max(best_in, cross) % MOD


sol = Solution()
for arr, k, want in examples:
    print(sol.kConcatenationMaxSum(arr, k))
    assert sol.kConcatenationMaxSum(arr, k) == want

9
2
0


## 10. Vì sao đúng

Gọi `B = arr * k`, `S = sum(arr)`. Một đoạn khác rỗng của `B` bắt đầu ở bản sao `p`, kết thúc ở bản sao `q` (`p ≤ q`).

1. **`p = q`:** đoạn nằm trong một bản → tổng `≤ best_in`.
2. **`p < q`:** đoạn = (hậu tố của bản `p`) + `(q − p − 1)` bản đầy đủ + (tiền tố của bản `q`). Tổng `≤ suf + pre + (q − p − 1)·S`. Vì `0 ≤ q − p − 1 ≤ k − 2`, giá trị này `≤ suf + pre + (k − 2)·max(S, 0)`. Và giá trị đó **đạt được** (chọn `p = 1, q = k` nếu `S > 0`, hoặc `q = p + 1` nếu `S ≤ 0`; hậu tố/tiền tố rỗng vẫn cho đoạn hợp lệ hoặc đoạn rỗng tổng 0).
3. Đoạn rỗng có tổng `0 ≤ best_in`.

Vậy đáp án `= max(best_in, suf + pre + (k−2)·max(S,0))` với `k ≥ 2`. Đây cũng chính là `kadane0(arr*2) + (k−2)·max(S,0)` (Cách 2), vì khi `S > 0`: `suf + pre ≥ best_in`? — với đoạn `arr[l..r]` tốt nhất, `suf ≥ sum(arr[l..n-1]) = sum(arr[l..r]) + sum(arr[r+1..])` và `pre ≥ sum(arr[0..r])`... cộng lại `suf + pre ≥ sum(arr[l..r]) + S ≥ best_in` khi `S > 0`. ✓ (Bộ test ngẫu nhiên bên dưới cũng xác nhận hai cách trùng nhau.)

**Modulo:** chỉ lấy ở cuối, sau khi đã so sánh `max` trên số nguyên thật.

## 11. Time / Space Complexity

Gọi `n = len(arr)`.

| Cách | Time | Space |
|---|---|---|
| 1. Brute (xây `arr*k`) | `O(n·k)` | `O(n·k)` |
| 2. Kadane trên 2 bản | `O(n)` | `O(n)` |
| 3. Tiền tố / hậu tố | `O(n)` | `O(1)` |
| 4. Chỉ số ảo `i % n` | `O(n)` | `O(1)` |

Không phụ thuộc `k` (trừ một phép nhân).

## 12. Bộ test

In [15]:
solvers = {
    "kadane2": solve_kadane2,
    "prefix_suffix": solve_prefix_suffix,
    "virtual": solve_virtual,
    "submit": Solution().kConcatenationMaxSum,
}
edge = [
    ([5], 1, 5),
    ([5], 100000, 500000),
    ([-5], 3, 0),
    ([0, 0], 4, 0),
    ([3, -4, 3], 1, 3),
    ([3, -4, 3], 2, 6),
    ([3, -4, 3], 3, 8),
    ([-1, 5, -1], 3, 11),
]
for arr, k, want in examples + edge:
    assert brute(arr, k) == want, (arr, k, brute(arr, k))
    for name, f in solvers.items():
        assert f(arr, k) == want, (name, arr, k, f(arr, k), want)
print("ví dụ + edge: OK")

ví dụ + edge: OK


In [16]:
random.seed(1191)
for trial in range(1500):
    n = random.randint(1, 8)
    k = random.randint(1, 6)
    arr = [random.randint(-6, 6) for _ in range(n)]
    want = brute_raw(arr, k) if n * k <= 20 else brute(arr, k)
    assert brute(arr, k) == want
    for name, f in solvers.items():
        assert f(arr, k) == want, (name, arr, k)
print("1500 test ngẫu nhiên khớp brute force")

1500 test ngẫu nhiên khớp brute force


In [17]:
random.seed(9)
big = [random.randint(-10**4, 10**4) for _ in range(10**5)]
for name, f in solvers.items():
    t = time.perf_counter(); r = f(big, 10**5); print(f"{name:14s} {r} {time.perf_counter() - t:.3f}s")
allmax = [10**4] * 10**5
expected = (10**4 * 10**5 * 10**5) % MOD
assert Solution().kConcatenationMaxSum(allmax, 10**5) == expected
print("test giới hạn OK, tổng thật =", 10**4 * 10**5 * 10**5, "-> mod =", expected)

kadane2        763799304 0.014s
prefix_suffix  763799304 0.015s
virtual        763799304 0.018s
submit         763799304 0.013s
test giới hạn OK, tổng thật = 100000000000000 -> mod = 999300007


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Luôn cộng `(k−2)·S` kể cả khi `S < 0`

In [18]:
def wrong_always_add(arr, k):
    if k == 1:
        return kadane0(arr) % MOD
    return (kadane0(arr * 2) + (k - 2) * sum(arr)) % MOD


print(wrong_always_add([3, -10, 3], 4), "đúng là", brute([3, -10, 3], 4))

1000000005 đúng là 6


**Vì sao sai:** `S = -4 < 0` nên thêm bản sao chỉ làm giảm; kết quả âm rồi modulo thành số khổng lồ. **Sửa:** `(k − 2)·max(S, 0)`.

### Lỗi 2 — Quên trường hợp `k == 1` (dùng `arr * 2` vô điều kiện)

In [19]:
def wrong_k1(arr, k):
    return (kadane0(arr * 2) + (k - 2) * max(sum(arr), 0)) % MOD


print(wrong_k1([1, -2, 1], 1), "đúng là", brute([1, -2, 1], 1))
print(wrong_k1([1, 2], 1), "đúng là", brute([1, 2], 1))

2 đúng là 1
3 đúng là 3


**Vì sao sai:** với `k = 1` không có bản thứ hai để "vắt qua", và `(k − 2) = −1` làm trừ đi `S`. Có trường hợp tình cờ đúng (`S > 0` thì `kadane0(arr*2) − S` có thể bằng `kadane0(arr)`), nhưng `[1, -2, 1]` cho `2` thay vì `1`. **Sửa:** xử lý `k == 1` riêng.

### Lỗi 3 — Lấy modulo giữa chừng rồi so sánh `max`

In [20]:
def wrong_mod_early(arr, k):
    best_in = kadane0(arr) % MOD
    pre = max(0, max(accumulate(arr)))
    suf = max(0, max(accumulate(reversed(arr))))
    cross = (suf + pre + (k - 2) * max(sum(arr), 0)) % MOD     # đã mod
    return max(best_in, cross) if k >= 2 else best_in


arr2 = [10**4] * 10**5          # best_in = 10^9, pre = suf = 10^9, S = 10^9
k = 2
print("sai :", wrong_mod_early(arr2, k))
print("đúng:", Solution().kConcatenationMaxSum(arr2, k))
print("best_in thật =", kadane0(arr2), "| cross thật =", 2 * 10**9, "-> cross % MOD =", 2 * 10**9 % MOD)

sai : 1000000000
đúng: 999999993
best_in thật = 1000000000 | cross thật = 2000000000 -> cross % MOD = 999999993


**Vì sao sai:** đáp án thật là `cross = 2·10^9` (cả hai bản), lấy modulo được `999999993`. Nhưng bản sai mod `cross` **trước** khi so sánh: `999999993 < best_in = 10^9`, nên nó chọn nhầm `best_in` và trả `10^9`. Modulo không bảo toàn thứ tự lớn/nhỏ. **Sửa:** so sánh trên số nguyên thật, chỉ mod ở `return`.

### Lỗi 4 — Kadane không cho phép rỗng

In [21]:
def wrong_nonempty(arr, k):
    a = arr * min(k, 2)
    cur = best = a[0]
    for x in a[1:]:
        cur = max(x, cur + x); best = max(best, cur)
    if k >= 2:
        best += (k - 2) * max(sum(arr), 0)
    return best % MOD


print(wrong_nonempty([-1, -2], 7), "đúng là", brute([-1, -2], 7))

1000000006 đúng là 0


**Vì sao sai:** đề cho phép đoạn rỗng (tổng `0`), Kadane "không rỗng" trả `-1` rồi mod thành `10^9 + 6`. **Sửa:** `cur = max(0, cur + x)`, `best` khởi tạo `0`.

## 14. Mẹo & nhận dạng pattern

- **"Mảng lặp lại `k` lần", "mảng vòng"** → xét **2 bản sao** là đủ để bao mọi đoạn "vắt qua chỗ nối"; phần lặp ở giữa đóng góp `m·S`.
- **Đoạn tối ưu qua nhiều khối** = hậu tố khối đầu + các khối đầy đủ + tiền tố khối cuối.
- **Kadane cho phép rỗng** vs **không rỗng**: khác ở khởi tạo và `max(0, …)`.
- **Modulo + max/min:** so sánh trên giá trị thật, mod ở cuối.
- **Thực nghiệm để tìm công thức:** chạy brute trên `k = 1..6`, nhìn hiệu.

## 15. Biến thể

1. **LeetCode 53 — Maximum Subarray** (`k = 1`, không rỗng).
2. **LeetCode 918 — Maximum Sum Circular Subarray:** mảng vòng, đoạn dài tối đa `n`.
3. **Đoạn có độ dài tối đa `L`** trên mảng lặp → cần deque/prefix-min trong cửa sổ.
4. **Tổng nhỏ nhất** trên mảng lặp → đổi dấu mảng rồi áp dụng cùng công thức.
5. **`k` cực lớn (`10^18`)** → công thức vẫn `O(n)` vì chỉ có một phép nhân `(k−2)·S`.

In [22]:
def k_concat_min_sum(arr, k):
    """Biến thể 4: tổng nhỏ nhất (cho phép rỗng) = - (max trên mảng đổi dấu)."""
    neg = [-x for x in arr]
    best_in = kadane0(neg)
    if k == 1:
        return -best_in
    pre = max(0, max(accumulate(neg)))
    suf = max(0, max(accumulate(reversed(neg))))
    return -max(best_in, suf + pre + (k - 2) * max(sum(neg), 0))


def brute_min(arr, k):
    b = arr * k; P = list(accumulate(b, initial=0))
    return min(0, min(P[r] - P[l] for l in range(len(b)) for r in range(l + 1, len(b) + 1)))


random.seed(15)
for _ in range(300):
    a = [random.randint(-5, 5) for _ in range(random.randint(1, 5))]
    kk = random.randint(1, 4)
    assert k_concat_min_sum(a, kk) == brute_min(a, kk)
print(k_concat_min_sum([1, -3, 1], 4), "| biến thể 4 OK; k = 10^18:", Solution().kConcatenationMaxSum([1, 2], 10**18))

-6 | biến thể 4 OK; k = 10^18: 147


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute | Xây `arr*k`, Kadane | `O(n·k)` | `O(n·k)` | Oracle với input nhỏ |
| 2. Kadane 2 bản | `kadane0(arr*2) + (k−2)·max(S,0)` | `O(n)` | `O(n)` | Nhanh, dễ nhớ |
| 3. Tiền tố/hậu tố | `max(best_in, suf+pre+(k−2)·max(S,0))` | `O(n)` | `O(1)` | Nộp bài, dễ chứng minh |
| 4. Chỉ số ảo | Kadane trên `i % n` | `O(n)` | `O(1)` | Khi không muốn cấp phát |
| 5. Thực nghiệm | Nhìn hiệu Kadane theo `k` | — | — | Khám phá công thức |

## 17. Bài tập vận dụng

**Bài 1 (đoán output):** `kConcatenationMaxSum([2, -1, 2], 3)` = ?

**Bài 2 (tính tay):** với `arr = [-3, 4, -2, 1]`, tính `best_in`, `pre`, `suf`, `S`, rồi đáp án cho `k = 1, 2, 10`.

**Bài 3 (điền TODO):** hoàn thiện `best_prefix(arr)` — tổng tiền tố lớn nhất (cho phép rỗng).

**Bài 4 (tự viết lại):** viết lời giải chỉ dùng **một vòng lặp** duy nhất trên `arr` (gợi ý: hậu tố lớn nhất = `S − min tiền tố`).

**Bài 5 (mini challenge):** trả về thêm **độ dài** của đoạn đạt tổng lớn nhất (nếu nhiều đoạn, lấy đoạn ngắn nhất).

In [23]:
def best_prefix(arr):
    # TODO: max(0, max tổng arr[0..i])
    pass


def one_pass(arr, k):
    # TODO: một vòng lặp duy nhất
    raise NotImplementedError


def best_with_length(arr, k):
    # TODO: trả về (tổng % MOD, độ dài ngắn nhất đạt tổng đó) -- gợi ý: dùng brute với k nhỏ để kiểm tra
    raise NotImplementedError

# assert best_prefix([1, -2, 3]) == 2
# assert one_pass([1, -2, 1], 5) == 2

## 18. Gợi ý 3 mức

- **Mức 1:** Đoạn tối ưu không cần quá 2 bản sao nếu tổng một bản `≤ 0`.
- **Mức 2:** Nếu tổng `> 0`, lấy hậu tố tốt nhất của bản đầu + tất cả bản giữa + tiền tố tốt nhất của bản cuối.
- **Mức 3:** Bài 4: hậu tố `arr[i..]` có tổng `S − P[i]` với `P[i]` là tiền tố trước `i`; hậu tố lớn nhất `= S − min(P[0..n])` (với `P[0] = 0`, `P[n] = S` cho hậu tố rỗng). Chỉ cần theo dõi `min` tiền tố trong cùng vòng lặp.

## 19. Đáp án tham khảo

**Bài 1:** `S = 3 > 0`, `kadane0(arr*2) = 2−1+2+2−1+2 = 6`, cộng `(3−2)·3 = 3` → `9` (cả mảng).

**Bài 2:** `best_in = 4` (`[4]`), `pre = 1` (`[-3,4]`), `suf = 3` (`[4,-2,1]`), `S = 0`. `k = 1 → 4`; `k = 2 → max(4, 3+1) = 4`; `k = 10 → 4`.

In [24]:
assert Solution().kConcatenationMaxSum([2, -1, 2], 3) == 9
assert [Solution().kConcatenationMaxSum([-3, 4, -2, 1], k) for k in (1, 2, 10)] == [4, 4, 4]


def best_prefix(arr):
    best = run = 0
    for x in arr:
        run += x
        best = max(best, run)
    return best


def one_pass(arr, k):
    best_in = cur = 0
    run = pre = 0
    min_p = 0                       # min tiền tố P[0..i], P[0] = 0
    for x in arr:
        cur = max(0, cur + x); best_in = max(best_in, cur)
        run += x
        pre = max(pre, run)
        min_p = min(min_p, run)
    S = run
    suf = S - min(min_p, S)         # S - min(P[0..n]); P[n] = S cho hậu tố rỗng
    if k == 1:
        return best_in % MOD
    return max(best_in, suf + pre + (k - 2) * max(S, 0)) % MOD


def best_with_length(arr, k):
    b = arr * k
    P = list(accumulate(b, initial=0))
    best, length = 0, 0
    for l in range(len(b)):
        for r in range(l + 1, len(b) + 1):
            s = P[r] - P[l]
            if s > best or (s == best and best > 0 and r - l < length):
                best, length = s, r - l
    return best % MOD, length


assert best_prefix([1, -2, 3]) == 2
random.seed(19)
for _ in range(1000):
    a = [random.randint(-6, 6) for _ in range(random.randint(1, 7))]
    kk = random.randint(1, 5)
    assert one_pass(a, kk) == brute(a, kk), (a, kk)
print(best_with_length([1, -2, 1], 5), best_with_length([1, 2], 3))
print("Đáp án bài tập: OK")

(2, 2) (9, 6)
Đáp án bài tập: OK


## 20. Template nhớ nhanh + Checklist trước khi submit

```python
best_in = kadane_cho_phep_rong(arr)
if k == 1: return best_in % MOD
pre = max tiền tố (>= 0); suf = max hậu tố (>= 0); S = sum(arr)
return max(best_in, suf + pre + (k - 2) * max(S, 0)) % MOD
# tương đương: (kadane0(arr * 2) + (k - 2) * max(S, 0)) % MOD
```

**Checklist:**
- [ ] Không xây mảng `arr * k`.
- [ ] `k == 1` xử lý riêng.
- [ ] Chỉ cộng `(k−2)·S` khi `S > 0`.
- [ ] Kadane **cho phép rỗng** (đáp án `≥ 0`).
- [ ] Modulo **chỉ ở cuối**, sau khi so sánh `max`.
- [ ] Test `k` lớn, mảng toàn dương, toàn âm, tổng bằng 0.